# Notebook 01: Path Planning — A* và Trajectory Generation

**Mục tiêu học tập:**
- Hiểu cách biểu diễn grid map 2D
- Triển khai và trực quan hóa thuật toán A*
- Chuyển discrete path thành trajectory có thời gian

**Nội dung:**
1. Load và trực quan hóa grid map
2. Chạy A* tìm đường
3. Sinh trajectory từ path
4. Smoothing trajectory

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from planner.astar import astar, load_map, path_to_cells
from planner.trajectory import path_to_trajectory, smooth_trajectory

## 1. Load và trực quan hóa Grid Map

In [ ]:
# Load empty map
m = load_map('../maps/empty_20x20.yaml')
print(f"Map: {m['name']}, size: {m['height']}x{m['width']}, resolution: {m['resolution']}m/cell")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, map_file, title in zip(axes,
    ['empty_20x20.yaml', 'obstacles_20x20.yaml', 'corridor_30x10.yaml'],
    ['Empty 20x20', 'Obstacles 20x20', 'Corridor 30x10']):
    m = load_map(f'../maps/{map_file}')
    ax.imshow(m['grid'], cmap='gray_r', origin='upper')
    ax.set_title(title)
    ax.set_xlabel('Col')
    ax.set_ylabel('Row')
plt.tight_layout()
plt.show()

## 2. Chạy A* tìm đường

In [ ]:
# Run A* on empty map
m = load_map('../maps/empty_20x20.yaml')
start, goal = (2, 2), (17, 17)

path_cells = astar(m['grid'], start, goal)
print(f"Path found: {len(path_cells)} cells")
print(f"Start: {path_cells[0]}, Goal: {path_cells[-1]}")

# Visualize
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(m['grid'], cmap='gray_r', origin='upper')
path_arr = np.array(path_cells)
ax.plot(path_arr[:, 1], path_arr[:, 0], 'r-', linewidth=2, label='A* Path')
ax.plot(start[1], start[0], 'go', markersize=10, label='Start')
ax.plot(goal[1], goal[0], 'r*', markersize=15, label='Goal')
ax.legend()
ax.set_title('A* Path on Empty Map')
plt.show()

In [ ]:
# Run A* on obstacles map
m = load_map('../maps/obstacles_20x20.yaml')
start, goal = (1, 1), (18, 18)

path_cells = astar(m['grid'], start, goal)
print(f"Path found: {len(path_cells)} cells")

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(m['grid'], cmap='gray_r', origin='upper')
path_arr = np.array(path_cells)
ax.plot(path_arr[:, 1], path_arr[:, 0], 'r-', linewidth=2)
ax.plot(start[1], start[0], 'go', markersize=10)
ax.plot(goal[1], goal[0], 'r*', markersize=15)
ax.set_title('A* Path with Obstacles')
plt.show()

## 3. Sinh Trajectory từ Path

In [ ]:
# Convert to world coordinates
m = load_map('../maps/empty_20x20.yaml')
path_cells = astar(m['grid'], (2, 2), (17, 17))
path_xy = path_to_cells(path_cells, resolution=m['resolution'])

# Generate trajectory
traj = path_to_trajectory(path_xy, dt=0.1, target_speed=1.0)
print(f"Trajectory: {len(traj)} points, duration: {traj['t'].iloc[-1]:.1f}s")
traj.head(10)

In [ ]:
# Plot trajectory with yaw arrows
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(traj['x_ref'], traj['y_ref'], 'b-o', markersize=2, label='Trajectory')
# Plot yaw arrows every 20 points
for i in range(0, len(traj), 20):
    dx = 0.3 * np.cos(traj['yaw_ref'].iloc[i])
    dy = 0.3 * np.sin(traj['yaw_ref'].iloc[i])
    ax.arrow(traj['x_ref'].iloc[i], traj['y_ref'].iloc[i], dx, dy,
             head_width=0.1, color='red')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Trajectory with Heading')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(traj['t'], traj['v_ref'], 'g-')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Speed (m/s)')
ax.set_title('Speed Profile')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Smoothing Trajectory

In [ ]:
smoothed = smooth_trajectory(traj, window=7)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(traj['x_ref'], traj['y_ref'], 'b-', alpha=0.5, label='Original')
ax.plot(smoothed['x_ref'], smoothed['y_ref'], 'r-', linewidth=2, label='Smoothed')
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Trajectory Smoothing')
ax.legend()
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

## Tóm tắt

- Grid map biểu diễn môi trường 2D với ô trống (0) và vật cản (1)
- A* tìm đường ngắn nhất trên grid, hỗ trợ 4/8-connected
- Trajectory chuyển discrete path thành tham chiếu thời gian (t, x, y, yaw, v)
- Smoothing giúp trajectory mượt hơn, phù hợp cho MPC

**Bài tập:** Thử tạo map mới, thay đổi start/goal, và quan sát ảnh hưởng của `target_speed` đến trajectory.